# Step 2 (Kaggle): physics features, flood weighting and future rainfall

Run with **Save Version → Save & Run All (Commit)**; it keeps running after you close the browser (about 3–3.5 hours). Results appear in the **Output** tab.

Settings (right panel): **Accelerator: GPU T4 x2** (one GPU is used), **Internet: On**.

**What is tested**

| Model | Extra inputs | Flood weighting |
|---|---|---|
| LSTM_BASE | none (same as Step 1) | no |
| LSTM_PHYS | antecedent rain (3/7-day totals, API) + Curve Number runoff | no |
| LSTM_PEAK | none | yes |
| LSTM_FRAIN | next 7 days of rainfall | no |
| LSTM_ALL | all of the above | yes |
| TFT_PHYS | antecedent rain + Curve Number runoff | yes |
| TFT_V2 | antecedent rain + Curve Number runoff + future rainfall | yes |

Future rainfall here is the observed rain of the next days (a perfect forecast), so FRAIN / ALL / TFT_V2 are an upper bound. TFT_PHYS is the realistic model that uses only past data.

In [ ]:
import subprocess, sys, os

LOG = '/kaggle/working/results/run_log.txt'
os.makedirs('/kaggle/working/results', exist_ok=True)

def run(cmd):
    """Run a shell command, show its output live, save it to the log, and stop if it fails."""
    print(f'\n$ {cmd}', flush=True)
    with open(LOG, 'a') as log:
        log.write(f'\n$ {cmd}\n')
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line)
        p.wait()
    if p.returncode != 0:
        raise RuntimeError(f'Step failed: {cmd} (see the output above or results/run_log.txt)')

run('nvidia-smi -L || echo no GPU')
run('cd /kaggle/working && (test -d flood || git clone -q https://github.com/yatharth7115/flood.git) && cd flood && git pull -q && git log --oneline -1')
os.chdir('/kaggle/working/flood')
run('pip install -q pytorch-forecasting pyyaml tabulate')

In [ ]:
import yaml
cfg = yaml.safe_load(open('configs/step2.yaml'))
cfg['data']['data_dir'] = '/tmp/camels_data'
cfg['data']['processed_file'] = '/tmp/camels_data/processed.parquet'
cfg['results_dir'] = '/kaggle/working/results'
os.makedirs('/tmp/camels_data', exist_ok=True)
yaml.safe_dump(cfg, open('configs/run.yaml', 'w'))
print('peak quantile:', cfg['peak_quantile'])

## 1. Data

In [ ]:
run('python src/download.py configs/run.yaml')
run('python src/dataset.py configs/run.yaml')

## 2. LSTM ablation (about 7 minutes each)

In [ ]:
LSTM_VARIANTS = [
    ('LSTM_BASE',  '',                               0),
    ('LSTM_PHYS',  'antecedent cn',                  0),
    ('LSTM_PEAK',  '',                               4),
    ('LSTM_FRAIN', 'future_rain',                    0),
    ('LSTM_ALL',   'antecedent cn future_rain',      4),
]
for name, extra, pw in LSTM_VARIANTS:
    ex = f'--extra {extra}' if extra else ''
    run(f'python src/train_lstm.py configs/run.yaml --name {name} {ex} --peak-weight {pw}')

## 3. TFT: realistic (past data only) and with future rainfall (about 1.3 hours each)

In [ ]:
run('python src/train_tft.py configs/run.yaml --name TFT_PHYS --extra antecedent cn --peak-weight 4')

In [ ]:
run('python src/train_tft.py configs/run.yaml --name TFT_V2 --extra antecedent cn future_rain --peak-weight 4')

## 4. Compare (with Step 1 results as reference)

In [ ]:
run('python src/evaluate.py configs/run.yaml --reference results/step1/metrics_per_gauge.csv --plot LSTM_BASE TFT_PHYS TFT_V2')
from IPython.display import Image, display, Markdown
R = cfg['results_dir']
display(Markdown(open(f'{R}/summary.md').read()))
for f in ['nse_cdf.png', 'hydrograph_focus.png', 'tft_phys_variable_importance.png', 'tft_v2_variable_importance.png']:
    if os.path.exists(f'{R}/{f}'):
        display(Image(f'{R}/{f}', width=700))